In [14]:
from pathlib import Path
import pandas as pd
from cyvcf2 import VCF
from tqdm import tqdm

In [15]:
snp_only_dir = Path("./snp_only")

vcf_files = sorted(snp_only_dir.glob("*.vcf"))

In [16]:
# Collect all unique SNP sites across samples 
all_sites = set()
sample_sites = {}

print("🔍 Reading SNP sites from all VCFs...")
for vcf_file in tqdm(vcf_files):
    sample_name = vcf_file.stem.replace("_snps_only", "")
    reader = VCF(str(vcf_file))
    sites = []
    for record in reader:
        # Use (CHROM, POS, REF, ALT) tuple for uniqueness
        for alt in record.ALT:
            site = (record.CHROM, record.POS, record.REF, str(alt))
            sites.append(site)
            all_sites.add(site)
    sample_sites[sample_name] = sites

🔍 Reading SNP sites from all VCFs...


100%|██████████| 16/16 [00:00<00:00, 32.93it/s]


In [17]:
all_sites = sorted(all_sites)

print(f"✅ Total unique SNP sites: {len(all_sites)}")

✅ Total unique SNP sites: 1446


In [18]:
matrix = []
for sample_name, sites in tqdm(sample_sites.items(), desc="🧬 Building SNP matrix"):
    sites_set = set(sites)
    row = [1 if site in sites_set else 0 for site in all_sites]
    matrix.append(row)

🧬 Building SNP matrix: 100%|██████████| 16/16 [00:00<00:00, 1331.39it/s]


In [19]:
# siguro next time, paki-add yung country and all huhu kung need
columns = [f"{chrom}_{pos}_{ref}_{alt}" for chrom, pos, ref, alt in all_sites]
df = pd.DataFrame(matrix, columns=columns, index=sample_sites.keys())

print("✅ SNP Matrix shape:", df.shape)

✅ SNP Matrix shape: (16, 1446)


In [20]:
# display DataFrame
display(df)

,Chromosome_16_G_T,Chromosome_135_G_A,Chromosome_211_C_T,Chromosome_516_C_A,Chromosome_962_G_C,Chromosome_1232_T_C,Chromosome_1247_T_A,Chromosome_5121_A_T,Chromosome_5385_T_G,Chromosome_5560_C_A,...,Chromosome_4407850_C_T,Chromosome_4407873_C_A,Chromosome_4407907_T_C,Chromosome_4407927_T_G,Chromosome_4408057_A_C,Chromosome_4408065_G_A,Chromosome_4408156_A_C,Chromosome_4408165_C_A,Chromosome_4408213_G_A,Chromosome_4408479_G_A
ERR2516329,1,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
ERR2516383,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
ERR4810602,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
ERR4810657,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
ERR4811220,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
ERR4830728,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ERR5917676,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,1,0
ERR5917689,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
SRR6152660,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
SRR6153217,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [21]:
df.to_csv("snp_matrix.csv")
print("📁 Saved SNP matrix to snp_matrix.csv")

📁 Saved SNP matrix to snp_matrix.csv


In [22]:
# test ko lang if keri na to input sa ml model ganern
check = pd.read_csv('/mnt/c/Users/Lenovo/Downloads/ths-st1-tb/snp_matrix.csv')
check

,Unnamed: 0,Chromosome_16_G_T,Chromosome_135_G_A,Chromosome_211_C_T,Chromosome_516_C_A,Chromosome_962_G_C,Chromosome_1232_T_C,Chromosome_1247_T_A,Chromosome_5121_A_T,Chromosome_5385_T_G,...,Chromosome_4407850_C_T,Chromosome_4407873_C_A,Chromosome_4407907_T_C,Chromosome_4407927_T_G,Chromosome_4408057_A_C,Chromosome_4408065_G_A,Chromosome_4408156_A_C,Chromosome_4408165_C_A,Chromosome_4408213_G_A,Chromosome_4408479_G_A
0,ERR2516329,1,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
1,ERR2516383,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
2,ERR4810602,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,ERR4810657,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,ERR4811220,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
5,ERR4830728,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,ERR5917676,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,1,0
7,ERR5917689,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
8,SRR6152660,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
9,SRR6153217,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
